
# Risk & Fraud Statistical Analysis

## Objective

Investigate insurance risk scores, fraud-flag patterns,
claim severity, and their relationships using Python
and statistical analysis.

The analysis aims to identify risk concentrations,
potential fraud indicators, and actionable insights
for insurance risk management.

**Note:** Fraud flags represent potential concerns
requiring investigation, not confirmed fraudulent activity.


In [3]:

# Import Required Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sqlalchemy import create_engine, URL
from getpass import getpass

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("All required libraries imported successfully.")


All required libraries imported successfully.


In [4]:

# Step 3 — MySQL Database Connection

db_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=getpass("Enter MySQL Password: "),
    host="localhost",
    port=3306,
    database="insurance_claims_analytics"
)

engine = create_engine(db_url)

# Test Database Connection
with engine.connect() as connection:
    print("MySQL database connected successfully.")


MySQL database connected successfully.


In [5]:

# Step 4 — Load Required Tables

claims = pd.read_sql("SELECT * FROM claims", engine)

risk_assessment = pd.read_sql(
    "SELECT * FROM risk_assessment", engine
)

policies = pd.read_sql("SELECT * FROM policies", engine)

customers = pd.read_sql("SELECT * FROM customers", engine)

# Verify Row Counts and Column Names
for name, df in {
    "Claims": claims,
    "Risk Assessment": risk_assessment,
    "Policies": policies,
    "Customers": customers
}.items():
    print(f"\n{name}: {len(df):,} rows")
    print("Columns:", df.columns.tolist())



Claims: 25,000 rows
Columns: ['Claim_ID', 'Policy_ID', 'Claim_Date', 'Claim_Amount', 'Claim_Status', 'Fraud_Flag', 'Claim_Type']

Risk Assessment: 10,000 rows
Columns: ['Customer_ID', 'Risk_Score', 'Risk_Category', 'Assessment_Date']

Policies: 15,000 rows
Columns: ['Policy_ID', 'Customer_ID', 'Policy_Type', 'Start_Date', 'End_Date', 'Annual_Premium', 'Coverage_Amount', 'Policy_Status']

Customers: 10,000 rows
Columns: ['Customer_ID', 'Age', 'Gender', 'Region', 'Join_Date', 'Annual_Income']


In [6]:

# BR 1 — Customer Risk Score Distribution

risk_distribution = (
    risk_assessment.groupby("Risk_Category")
    .agg(
        Total_Customers=("Customer_ID", "nunique"),
        Average_Risk_Score=("Risk_Score", "mean"),
        Median_Risk_Score=("Risk_Score", "median"),
        Minimum_Risk_Score=("Risk_Score", "min"),
        Maximum_Risk_Score=("Risk_Score", "max")
    )
    .reset_index()
)

risk_distribution["Customer_Share_Pct"] = (
    risk_distribution["Total_Customers"]
    / risk_assessment["Customer_ID"].nunique() * 100
)

risk_distribution = risk_distribution.sort_values(
    "Average_Risk_Score",
    ascending=False
)

display(risk_distribution.round(2))


,Risk_Category,Total_Customers,Average_Risk_Score,Median_Risk_Score,Minimum_Risk_Score,Maximum_Risk_Score,Customer_Share_Pct
0,High,1520,79.73,78.00,70,100,15.20
2,Medium,6132,51.36,51.00,35,69,61.32
1,Low,2348,22.95,25.00,0,34,23.48


In [7]:

# BR 2 — Fraud-Flag Rate by Customer Risk Category

claims_risk = (
    claims.merge(
        policies[["Policy_ID", "Customer_ID"]],
        on="Policy_ID",
        how="left",
        validate="many_to_one"
    )
    .merge(
        risk_assessment[
            ["Customer_ID", "Risk_Score", "Risk_Category"]
        ],
        on="Customer_ID",
        how="left",
        validate="many_to_one"
    )
)

fraud_by_risk = (
    claims_risk.groupby("Risk_Category")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum"),
        Average_Risk_Score=("Risk_Score", "mean")
    )
    .reset_index()
)

fraud_by_risk["Fraud_Flag_Rate_Pct"] = (
    fraud_by_risk["Fraud_Flagged_Claims"]
    / fraud_by_risk["Total_Claims"] * 100
)

fraud_by_risk = fraud_by_risk.sort_values(
    "Fraud_Flag_Rate_Pct",
    ascending=False
)

display(fraud_by_risk.round(2))

print("Total Claims:", fraud_by_risk["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    fraud_by_risk["Fraud_Flagged_Claims"].sum()
)


,Risk_Category,Total_Claims,Fraud_Flagged_Claims,Average_Risk_Score,Fraud_Flag_Rate_Pct
0,High,3962,385,79.76,9.72
2,Medium,15302,395,51.27,2.58
1,Low,5736,141,23.09,2.46


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [8]:

# BR 3 — Chi-Square Test: Risk Category vs Fraud Flag

contingency_table = pd.crosstab(
    claims_risk["Risk_Category"],
    claims_risk["Fraud_Flag"]
)

chi2, p_value, dof, expected = stats.chi2_contingency(
    contingency_table
)

n = contingency_table.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print("Contingency Table:")
display(contingency_table)

print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p_value:.6g}")
print(f"Degrees of Freedom: {dof}")
print(f"Cramer's V: {cramers_v:.4f}")
print(f"Minimum Expected Frequency: {expected.min():.2f}")

if p_value < 0.05:
    print("Result: Statistically significant association.")
else:
    print("Result: No statistically significant association.")


Contingency Table:


Fraud_Flag,0,1
Risk_Category,,
High,3577,385
Low,5595,141
Medium,14907,395


Chi-Square Statistic: 483.1749
P-Value: 1.20201e-105
Degrees of Freedom: 2
Cramer's V: 0.1390
Minimum Expected Frequency: 145.96
Result: Statistically significant association.


In [9]:

# BR 4 — Fraud-Flag Rate by Risk Score Band

risk_score_analysis = claims_risk.copy()

risk_score_analysis["Risk_Score_Band"] = pd.cut(
    risk_score_analysis["Risk_Score"],
    bins=[-1, 20, 40, 60, 80, 100],
    labels=["0–20", "21–40", "41–60", "61–80", "81–100"]
)

risk_band_fraud = (
    risk_score_analysis.groupby(
        "Risk_Score_Band", observed=True
    )
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum"),
        Average_Risk_Score=("Risk_Score", "mean")
    )
    .reset_index()
)

risk_band_fraud["Fraud_Flag_Rate_Pct"] = (
    risk_band_fraud["Fraud_Flagged_Claims"]
    / risk_band_fraud["Total_Claims"] * 100
)

display(risk_band_fraud.round(2))


,Risk_Score_Band,Total_Claims,Fraud_Flagged_Claims,Average_Risk_Score,Fraud_Flag_Rate_Pct
0,0–20,1929,43,12.44,2.23
1,21–40,6392,171,32.22,2.68
2,41–60,9509,236,50.37,2.48
3,61–80,5676,309,69.13,5.44
4,81–100,1494,162,88.12,10.84



### Key Finding — Risk Score and Fraud-Flag Patterns

Fraud-flag rates are substantially higher among claims associated
with elevated customer risk scores.

- **Highest fraud-flag rate:** 10.84% for scores 81–100.
- **Lowest fraud-flag rate:** 2.23% for scores 0–20.
- **Observed pattern:** Fraud-flag rates increase notably above
  a risk score of 60, although the relationship is not strictly
  increasing across every band.

**Business implication:** Higher-risk claims may warrant
prioritized investigation. Fraud flags are indicators for
review, not proof of fraudulent activity.


In [10]:

# BR 5 — Risk Score Comparison by Fraud Flag

risk_score_comparison = (
    claims_risk.groupby("Fraud_Flag")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Average_Risk_Score=("Risk_Score", "mean"),
        Median_Risk_Score=("Risk_Score", "median"),
        Risk_Score_Std_Dev=("Risk_Score", "std")
    )
    .reset_index()
)

display(risk_score_comparison.round(2))

# Separate risk scores by fraud-flag status
flagged_scores = claims_risk.loc[
    claims_risk["Fraud_Flag"] == 1, "Risk_Score"
].dropna()

non_flagged_scores = claims_risk.loc[
    claims_risk["Fraud_Flag"] == 0, "Risk_Score"
].dropna()

# Mann–Whitney U Test
u_statistic, p_value = stats.mannwhitneyu(
    flagged_scores,
    non_flagged_scores,
    alternative="two-sided"
)

print(f"\nMann–Whitney U Statistic: {u_statistic:,.2f}")
print(f"P-Value: {p_value:.6g}")

if p_value < 0.05:
    print("Result: Statistically significant difference.")
else:
    print("Result: No statistically significant difference.")


,Fraud_Flag,Total_Claims,Average_Risk_Score,Median_Risk_Score,Risk_Score_Std_Dev
0,0,24079,48.94,49.00,19.60
1,1,921,59.21,62.00,22.10



Mann–Whitney U Statistic: 14,148,522.50
P-Value: 5.30476e-46
Result: Statistically significant difference.



### Key Finding — Risk Score Differences by Fraud Flag

Fraud-flagged claims have higher customer risk scores than
non-flagged claims.

- **Average risk score:** 59.21 (flagged) vs. 48.94 (non-flagged).
- **Median risk score:** 62 vs. 49.
- **Mann–Whitney U:** 14,148,522.50.
- **P-value:** 5.30476 × 10⁻⁴⁶.

The Mann–Whitney U test indicates a statistically significant
difference in risk-score distributions (p < 0.05).

**Business implication:** Customer risk scores may help prioritize
claims for further review. However, fraud flags do not establish
confirmed fraud, and repeated customer observations limit the
independence assumption of the test.


In [12]:

# BR 6 — Fraud-Flag Exposure by Policy Type

fraud_policy_analysis = claims.merge(
    policies[["Policy_ID", "Policy_Type"]],
    on="Policy_ID",
    how="left",
    validate="many_to_one"
)

fraud_policy_analysis["Flagged_Claim_Amount"] = np.where(
    fraud_policy_analysis["Fraud_Flag"] == 1,
    fraud_policy_analysis["Claim_Amount"],
    0
)

fraud_by_policy = (
    fraud_policy_analysis.groupby("Policy_Type")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum"),
        Total_Flagged_Claim_Amount=("Flagged_Claim_Amount", "sum")
    )
    .reset_index()
)

fraud_by_policy["Fraud_Flag_Rate_Pct"] = (
    fraud_by_policy["Fraud_Flagged_Claims"]
    / fraud_by_policy["Total_Claims"] * 100
)

fraud_by_policy["Flagged_Amount_Share_Pct"] = (
    fraud_by_policy["Total_Flagged_Claim_Amount"]
    / fraud_by_policy["Total_Flagged_Claim_Amount"].sum() * 100
)

fraud_by_policy = fraud_by_policy.sort_values(
    "Fraud_Flag_Rate_Pct",
    ascending=False
)

display(fraud_by_policy.round(2))

print("Total Claims:", fraud_by_policy["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    fraud_by_policy["Fraud_Flagged_Claims"].sum()
)


,Policy_Type,Total_Claims,Fraud_Flagged_Claims,Total_Flagged_Claim_Amount,Fraud_Flag_Rate_Pct,Flagged_Amount_Share_Pct
2,Home,4458,172,"6,684,357.91",3.86,17.28
1,Health,7784,287,"12,001,861.49",3.69,31.03
0,Auto,9832,359,"15,951,295.53",3.65,41.24
3,Life,2926,103,"4,040,632.77",3.52,10.45


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [13]:

# BR 7 — Fraud-Flag Rate by Claim Status

fraud_by_status = (
    claims.groupby("Claim_Status")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum")
    )
    .reset_index()
)

fraud_by_status["Fraud_Flag_Rate_Pct"] = (
    fraud_by_status["Fraud_Flagged_Claims"]
    / fraud_by_status["Total_Claims"] * 100
)

fraud_by_status["Share_of_All_Flagged_Claims_Pct"] = (
    fraud_by_status["Fraud_Flagged_Claims"]
    / fraud_by_status["Fraud_Flagged_Claims"].sum() * 100
)

fraud_by_status = fraud_by_status.sort_values(
    "Fraud_Flag_Rate_Pct",
    ascending=False
)

display(fraud_by_status.round(2))

print("Total Claims:", fraud_by_status["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    fraud_by_status["Fraud_Flagged_Claims"].sum()
)


,Claim_Status,Total_Claims,Fraud_Flagged_Claims,Fraud_Flag_Rate_Pct,Share_of_All_Flagged_Claims_Pct
2,Rejected,4256,163,3.83,17.70
0,Approved,6100,230,3.77,24.97
1,Pending,4075,149,3.66,16.18
3,Settled,10569,379,3.59,41.15


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [14]:

# BR 8 — Fraud-Flag Rate by Claim Value Group

fraud_claim_value = claims.copy()

fraud_claim_value["Claim_Value_Group"] = np.where(
    fraud_claim_value["Claim_Amount"] > 100000,
    "High Value (>₹1 Lakh)",
    "Standard Value (≤₹1 Lakh)"
)

fraud_by_value = (
    fraud_claim_value.groupby("Claim_Value_Group")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum"),
        Total_Claimed_Amount=("Claim_Amount", "sum")
    )
    .reset_index()
)

fraud_by_value["Fraud_Flag_Rate_Pct"] = (
    fraud_by_value["Fraud_Flagged_Claims"]
    / fraud_by_value["Total_Claims"] * 100
)

fraud_by_value["Share_of_All_Flags_Pct"] = (
    fraud_by_value["Fraud_Flagged_Claims"]
    / fraud_by_value["Fraud_Flagged_Claims"].sum() * 100
)

display(fraud_by_value.round(2))

print("Total Claims:", fraud_by_value["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    fraud_by_value["Fraud_Flagged_Claims"].sum()
)


,Claim_Value_Group,Total_Claims,Fraud_Flagged_Claims,Total_Claimed_Amount,Fraud_Flag_Rate_Pct,Share_of_All_Flags_Pct
0,High Value (>₹1 Lakh),1861,69,"284,094,209.07",3.71,7.49
1,Standard Value (≤₹1 Lakh),23139,852,"769,555,392.38",3.68,92.51


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [15]:

# BR 9 — Fraud-Flag Rate by Risk Category and Claim Value

risk_value_analysis = claims_risk.copy()

risk_value_analysis["Claim_Value_Group"] = np.where(
    risk_value_analysis["Claim_Amount"] > 100000,
    "High Value",
    "Standard Value"
)

risk_value_summary = (
    risk_value_analysis.groupby(
        ["Risk_Category", "Claim_Value_Group"]
    )
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum")
    )
    .reset_index()
)

risk_value_summary["Fraud_Flag_Rate_Pct"] = (
    risk_value_summary["Fraud_Flagged_Claims"]
    / risk_value_summary["Total_Claims"] * 100
)

risk_value_summary = risk_value_summary.sort_values(
    ["Risk_Category", "Claim_Value_Group"]
)

display(risk_value_summary.round(2))

print("Total Claims:", risk_value_summary["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    risk_value_summary["Fraud_Flagged_Claims"].sum()
)


,Risk_Category,Claim_Value_Group,Total_Claims,Fraud_Flagged_Claims,Fraud_Flag_Rate_Pct
0,High,High Value,320,25,7.81
1,High,Standard Value,3642,360,9.88
2,Low,High Value,420,9,2.14
3,Low,Standard Value,5316,132,2.48
4,Medium,High Value,1121,35,3.12
5,Medium,Standard Value,14181,360,2.54


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [16]:

# BR 10 — Fraud-Flag Rate by Customer Region

fraud_region_analysis = claims_risk.merge(
    customers[["Customer_ID", "Region"]],
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

fraud_by_region = (
    fraud_region_analysis.groupby("Region")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum"),
        Average_Risk_Score=("Risk_Score", "mean")
    )
    .reset_index()
)

fraud_by_region["Fraud_Flag_Rate_Pct"] = (
    fraud_by_region["Fraud_Flagged_Claims"]
    / fraud_by_region["Total_Claims"] * 100
)

fraud_by_region["Share_of_All_Flags_Pct"] = (
    fraud_by_region["Fraud_Flagged_Claims"]
    / fraud_by_region["Fraud_Flagged_Claims"].sum() * 100
)

fraud_by_region = fraud_by_region.sort_values(
    "Fraud_Flag_Rate_Pct",
    ascending=False
)

display(fraud_by_region.round(2))

print("Total Claims:", fraud_by_region["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    fraud_by_region["Fraud_Flagged_Claims"].sum()
)


,Region,Total_Claims,Fraud_Flagged_Claims,Average_Risk_Score,Fraud_Flag_Rate_Pct,Share_of_All_Flags_Pct
4,West,5035,201,49.87,3.99,21.82
0,Central,5045,201,50.16,3.98,21.82
1,East,4997,181,48.73,3.62,19.65
2,North,5062,177,48.90,3.50,19.22
3,South,4861,161,48.92,3.31,17.48


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [17]:

# BR 11 — Fraud-Flag Rate by Customer Age Group

fraud_age_analysis = fraud_region_analysis.merge(
    customers[["Customer_ID", "Age"]],
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

fraud_age_analysis["Age_Group"] = pd.cut(
    fraud_age_analysis["Age"],
    bins=[17, 30, 45, 60, 80],
    labels=["18–30", "31–45", "46–60", "61–80"]
)

fraud_by_age = (
    fraud_age_analysis.groupby("Age_Group", observed=True)
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum"),
        Average_Risk_Score=("Risk_Score", "mean")
    )
    .reset_index()
)

fraud_by_age["Fraud_Flag_Rate_Pct"] = (
    fraud_by_age["Fraud_Flagged_Claims"]
    / fraud_by_age["Total_Claims"] * 100
)

fraud_by_age["Share_of_All_Flags_Pct"] = (
    fraud_by_age["Fraud_Flagged_Claims"]
    / fraud_by_age["Fraud_Flagged_Claims"].sum() * 100
)

display(fraud_by_age.round(2))

print("Total Claims:", fraud_by_age["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    fraud_by_age["Fraud_Flagged_Claims"].sum()
)


,Age_Group,Total_Claims,Fraud_Flagged_Claims,Average_Risk_Score,Fraud_Flag_Rate_Pct,Share_of_All_Flags_Pct
0,18–30,5196,203,48.52,3.91,22.04
1,31–45,5984,237,49.72,3.96,25.73
2,46–60,5981,203,49.72,3.39,22.04
3,61–80,7839,278,49.24,3.55,30.18


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [18]:

# BR 12 — Fraud-Flag Rate by Claim Type

fraud_by_claim_type = (
    claims.groupby("Claim_Type")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum")
    )
    .reset_index()
)

fraud_by_claim_type["Fraud_Flag_Rate_Pct"] = (
    fraud_by_claim_type["Fraud_Flagged_Claims"]
    / fraud_by_claim_type["Total_Claims"] * 100
)

fraud_by_claim_type["Share_of_All_Flags_Pct"] = (
    fraud_by_claim_type["Fraud_Flagged_Claims"]
    / fraud_by_claim_type["Fraud_Flagged_Claims"].sum() * 100
)

fraud_by_claim_type = fraud_by_claim_type.sort_values(
    "Fraud_Flag_Rate_Pct",
    ascending=False
)

display(fraud_by_claim_type.round(2))

print("Total Claims:", fraud_by_claim_type["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    fraud_by_claim_type["Fraud_Flagged_Claims"].sum()
)


,Claim_Type,Total_Claims,Fraud_Flagged_Claims,Fraud_Flag_Rate_Pct,Share_of_All_Flags_Pct
2,Home,4458,172,3.86,18.68
1,Health,7784,287,3.69,31.16
0,Auto,9832,359,3.65,38.98
3,Life,2926,103,3.52,11.18


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [19]:

# Check whether Claim_Type and Policy_Type contain identical values

claim_policy_check = claims.merge(
    policies[["Policy_ID", "Policy_Type"]],
    on="Policy_ID",
    how="left",
    validate="many_to_one"
)

matching_claims = (
    claim_policy_check["Claim_Type"]
    == claim_policy_check["Policy_Type"]
).sum()

mismatching_claims = (
    claim_policy_check["Claim_Type"]
    != claim_policy_check["Policy_Type"]
).sum()

print("Total Claims:", len(claim_policy_check))
print("Matching Claim and Policy Types:", matching_claims)
print("Mismatching Claim and Policy Types:", mismatching_claims)


Total Claims: 25000
Matching Claim and Policy Types: 25000
Mismatching Claim and Policy Types: 0


In [20]:

# BR 12 — Fraud-Flag Rate by Customer Income Group

fraud_income_analysis = claims_risk.merge(
    customers[["Customer_ID", "Annual_Income"]],
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

# Divide customers into four income groups
fraud_income_analysis["Income_Group"] = pd.qcut(
    fraud_income_analysis["Annual_Income"],
    q=4,
    labels=["Q1 - Lowest", "Q2", "Q3", "Q4 - Highest"]
)

fraud_by_income = (
    fraud_income_analysis.groupby("Income_Group", observed=True)
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum"),
        Average_Risk_Score=("Risk_Score", "mean")
    )
    .reset_index()
)

fraud_by_income["Fraud_Flag_Rate_Pct"] = (
    fraud_by_income["Fraud_Flagged_Claims"]
    / fraud_by_income["Total_Claims"] * 100
)

display(fraud_by_income.round(2))

print("Total Claims:", fraud_by_income["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    fraud_by_income["Fraud_Flagged_Claims"].sum()
)


,Income_Group,Total_Claims,Fraud_Flagged_Claims,Average_Risk_Score,Fraud_Flag_Rate_Pct
0,Q1 - Lowest,6252,225,48.63,3.60
1,Q2,6248,226,48.95,3.62
2,Q3,6254,217,50.82,3.47
3,Q4 - Highest,6246,253,48.87,4.05


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [21]:

# BR 13 — Fraud-Flag Rate by Customer Claim Frequency

# Count claims submitted by each customer
customer_claim_frequency = (
    claims_risk.groupby("Customer_ID")
    .agg(
        Customer_Claim_Count=("Claim_ID", "count")
    )
    .reset_index()
)

# Attach claim frequency to each claim
fraud_frequency_analysis = claims_risk.merge(
    customer_claim_frequency,
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

# Categorize customers based on claim frequency
fraud_frequency_analysis["Claim_Frequency_Group"] = pd.cut(
    fraud_frequency_analysis["Customer_Claim_Count"],
    bins=[0, 1, 3, 5, float("inf")],
    labels=["1 Claim", "2–3 Claims", "4–5 Claims", "6+ Claims"]
)

fraud_by_frequency = (
    fraud_frequency_analysis.groupby(
        "Claim_Frequency_Group", observed=True
    )
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum"),
        Unique_Customers=("Customer_ID", "nunique")
    )
    .reset_index()
)

fraud_by_frequency["Fraud_Flag_Rate_Pct"] = (
    fraud_by_frequency["Fraud_Flagged_Claims"]
    / fraud_by_frequency["Total_Claims"] * 100
)

display(fraud_by_frequency.round(2))

print("Total Claims:", fraud_by_frequency["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    fraud_by_frequency["Fraud_Flagged_Claims"].sum()
)


,Claim_Frequency_Group,Total_Claims,Fraud_Flagged_Claims,Unique_Customers,Fraud_Flag_Rate_Pct
0,1 Claim,1386,53,1386,3.82
1,2–3 Claims,6598,252,2709,3.82
2,4–5 Claims,7288,267,1643,3.66
3,6+ Claims,9728,349,1269,3.59


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [22]:

# BR 14 — Fraud-Flag Rate by Policy Tenure at Claim Date

fraud_tenure_analysis = claims.merge(
    policies[["Policy_ID", "Start_Date"]],
    on="Policy_ID",
    how="left",
    validate="many_to_one"
)

fraud_tenure_analysis["Claim_Date"] = pd.to_datetime(
    fraud_tenure_analysis["Claim_Date"]
)

fraud_tenure_analysis["Start_Date"] = pd.to_datetime(
    fraud_tenure_analysis["Start_Date"]
)

# Calculate policy age when the claim was submitted
fraud_tenure_analysis["Policy_Age_Days"] = (
    fraud_tenure_analysis["Claim_Date"]
    - fraud_tenure_analysis["Start_Date"]
).dt.days

# Identify claims occurring before policy commencement
invalid_tenure = (
    fraud_tenure_analysis["Policy_Age_Days"] < 0
).sum()

print("Claims Before Policy Start:", invalid_tenure)

# Group valid claims by policy age
valid_tenure = fraud_tenure_analysis[
    fraud_tenure_analysis["Policy_Age_Days"] >= 0
].copy()

valid_tenure["Policy_Tenure_Group"] = pd.cut(
    valid_tenure["Policy_Age_Days"],
    bins=[-1, 30, 90, 365, float("inf")],
    labels=["0–30 Days", "31–90 Days", "91–365 Days", "Over 1 Year"]
)

fraud_by_tenure = (
    valid_tenure.groupby("Policy_Tenure_Group", observed=True)
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum")
    )
    .reset_index()
)

fraud_by_tenure["Fraud_Flag_Rate_Pct"] = (
    fraud_by_tenure["Fraud_Flagged_Claims"]
    / fraud_by_tenure["Total_Claims"] * 100
)

display(fraud_by_tenure.round(2))

print("Valid Claims:", len(valid_tenure))
print(
    "Fraud-Flagged Valid Claims:",
    valid_tenure["Fraud_Flag"].sum()
)


Claims Before Policy Start: 0


,Policy_Tenure_Group,Total_Claims,Fraud_Flagged_Claims,Fraud_Flag_Rate_Pct
0,0–30 Days,1723,70,4.06
1,31–90 Days,3630,149,4.10
2,91–365 Days,15608,563,3.61
3,Over 1 Year,4039,139,3.44


Valid Claims: 25000
Fraud-Flagged Valid Claims: 921


In [23]:

# BR 15 — Fraud-Flag Rate by Coverage Utilization

fraud_coverage_analysis = claims.merge(
    policies[["Policy_ID", "Coverage_Amount"]],
    on="Policy_ID",
    how="left",
    validate="many_to_one"
)

# Calculate claim amount as a percentage of policy coverage
fraud_coverage_analysis["Coverage_Utilization_Pct"] = (
    fraud_coverage_analysis["Claim_Amount"]
    / fraud_coverage_analysis["Coverage_Amount"] * 100
)

# Categorize coverage utilization
fraud_coverage_analysis["Coverage_Utilization_Group"] = pd.cut(
    fraud_coverage_analysis["Coverage_Utilization_Pct"],
    bins=[-1, 10, 25, 50, 100, float("inf")],
    labels=[
        "0–10%",
        "10–25%",
        "25–50%",
        "50–100%",
        "Above 100%"
    ]
)

fraud_by_coverage = (
    fraud_coverage_analysis.groupby(
        "Coverage_Utilization_Group", observed=True
    )
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum"),
        Average_Utilization_Pct=("Coverage_Utilization_Pct", "mean")
    )
    .reset_index()
)

fraud_by_coverage["Fraud_Flag_Rate_Pct"] = (
    fraud_by_coverage["Fraud_Flagged_Claims"]
    / fraud_by_coverage["Total_Claims"] * 100
)

display(fraud_by_coverage.round(2))

print("Total Claims:", fraud_by_coverage["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    fraud_by_coverage["Fraud_Flagged_Claims"].sum()
)


,Coverage_Utilization_Group,Total_Claims,Fraud_Flagged_Claims,Average_Utilization_Pct,Fraud_Flag_Rate_Pct
0,0–10%,17578,637,4.02,3.62
1,10–25%,5153,196,15.42,3.80
2,25–50%,1620,68,33.95,4.20
3,50–100%,506,11,67.19,2.17
4,Above 100%,143,9,122.69,6.29


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [24]:

# BR 16 — Annual Fraud-Flag Trend

fraud_year_analysis = claims.copy()

fraud_year_analysis["Claim_Date"] = pd.to_datetime(
    fraud_year_analysis["Claim_Date"]
)

fraud_year_analysis["Claim_Year"] = (
    fraud_year_analysis["Claim_Date"].dt.year
)

fraud_by_year = (
    fraud_year_analysis.groupby("Claim_Year")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum")
    )
    .reset_index()
)

fraud_by_year["Fraud_Flag_Rate_Pct"] = (
    fraud_by_year["Fraud_Flagged_Claims"]
    / fraud_by_year["Total_Claims"] * 100
)

display(fraud_by_year.round(2))

print("Total Claims:", fraud_by_year["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    fraud_by_year["Fraud_Flagged_Claims"].sum()
)


,Claim_Year,Total_Claims,Fraud_Flagged_Claims,Fraud_Flag_Rate_Pct
0,2020,1639,55,3.36
1,2021,3723,136,3.65
2,2022,4099,176,4.29
3,2023,4163,137,3.29
4,2024,4376,172,3.93
5,2025,4316,156,3.61
6,2026,2684,89,3.32


Total Claims: 25000
Total Fraud-Flagged Claims: 921


In [25]:

# BR 17 — Chi-Square Test: Policy Tenure vs Fraud Flag

tenure_contingency = pd.crosstab(
    valid_tenure["Policy_Tenure_Group"],
    valid_tenure["Fraud_Flag"]
)

display(tenure_contingency)

# Chi-square test of independence
chi2, p_value, dof, expected = stats.chi2_contingency(
    tenure_contingency
)

# Calculate Cramer's V
n = tenure_contingency.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 / (n * (min(tenure_contingency.shape) - 1))
)

print(f"\nChi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p_value:.6g}")
print(f"Degrees of Freedom: {dof}")
print(f"Cramer's V: {cramers_v:.4f}")
print(f"Minimum Expected Frequency: {expected.min():.2f}")

if p_value < 0.05:
    print("Result: Statistically significant association.")
else:
    print("Result: No statistically significant association.")


Fraud_Flag,0,1
Policy_Tenure_Group,,
0–30 Days,1653,70
31–90 Days,3481,149
91–365 Days,15045,563
Over 1 Year,3900,139



Chi-Square Statistic: 3.4365
P-Value: 0.329094
Degrees of Freedom: 3
Cramer's V: 0.0117
Minimum Expected Frequency: 63.48
Result: No statistically significant association.


In [26]:

# BR 18 — Risk-Based Investigation Prioritization

risk_priority = (
    claims_risk.groupby("Risk_Category")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Fraud_Flagged_Claims=("Fraud_Flag", "sum")
    )
    .reset_index()
)

risk_priority["Review_Workload_Pct"] = (
    risk_priority["Total_Claims"]
    / risk_priority["Total_Claims"].sum() * 100
)

risk_priority["Flag_Capture_Pct"] = (
    risk_priority["Fraud_Flagged_Claims"]
    / risk_priority["Fraud_Flagged_Claims"].sum() * 100
)

risk_priority["Fraud_Flag_Rate_Pct"] = (
    risk_priority["Fraud_Flagged_Claims"]
    / risk_priority["Total_Claims"] * 100
)

risk_priority["Capture_to_Workload_Ratio"] = (
    risk_priority["Flag_Capture_Pct"]
    / risk_priority["Review_Workload_Pct"]
)

risk_priority["Risk_Category"] = pd.Categorical(
    risk_priority["Risk_Category"],
    categories=["High", "Medium", "Low"],
    ordered=True
)

risk_priority = risk_priority.sort_values("Risk_Category")

display(risk_priority.round(2))

print("Total Claims:", risk_priority["Total_Claims"].sum())
print(
    "Total Fraud-Flagged Claims:",
    risk_priority["Fraud_Flagged_Claims"].sum()
)


,Risk_Category,Total_Claims,Fraud_Flagged_Claims,Review_Workload_Pct,Flag_Capture_Pct,Fraud_Flag_Rate_Pct,Capture_to_Workload_Ratio
0,High,3962,385,15.85,41.80,9.72,2.64
2,Medium,15302,395,61.21,42.89,2.58,0.70
1,Low,5736,141,22.94,15.31,2.46,0.67


Total Claims: 25000
Total Fraud-Flagged Claims: 921



## Final Findings and Business Recommendations

### Key Findings

1. **Portfolio Fraud-Flag Rate**
   - Of 25,000 claims, 921 (3.68%) were fraud-flagged.
   - Fraud flags indicate claims requiring review, not confirmed fraud.

2. **Customer Risk and Fraud Flags**
   - High-risk customers had a 9.72% fraud-flag rate,
     compared with 2.58% for Medium-risk and 2.46% for Low-risk customers.
   - The Chi-square test found a statistically significant
     association between risk category and fraud flags
     (χ² = 483.17, p < 0.001).
   - Cramér's V = 0.139 indicates a weak association.

3. **Risk Score Statistical Comparison**
   - Fraud-flagged claims had a higher average customer
     risk score (59.21) than non-flagged claims (48.94).
   - The Mann–Whitney U test found a statistically
     significant difference (p < 0.001).

4. **Claim and Customer Characteristics**
   - Fraud-flag rates were relatively similar across
     policy types, claim statuses, regions, age groups,
     income groups, and claim-frequency categories.
   - High-value claims had a 3.71% flag rate, compared
     with 3.68% for standard-value claims.
   - Claims exceeding policy coverage had a 6.29% flag
     rate, but this group contained only 143 claims.

5. **Policy Tenure and Annual Patterns**
   - Newer policies showed slightly higher observed
     fraud-flag rates.
   - However, the policy-tenure Chi-square test found
     no statistically significant association
     (χ² = 3.44, p = 0.329).
   - Annual fraud-flag rates fluctuated without a
     consistent increasing or decreasing trend.

6. **Investigation Prioritization**
   - High-risk claims represented 15.85% of total
     claim-review workload.
   - This group accounted for 41.80% of all flagged claims.
   - The capture-to-workload ratio was 2.64×.

### Business Recommendations

1. Prioritize High-risk claims for additional investigation
   while maintaining fair and consistent review procedures.

2. Combine customer risk scores with relevant claim
   characteristics and policy-rule checks rather than
   relying on claim amount or demographics alone.

3. Review claims exceeding policy coverage for possible
   coverage-rule violations or data-quality issues.

4. Monitor fraud-flag rates and investigation outcomes
   regularly to evaluate review effectiveness.

5. Validate flagged cases against confirmed investigation
   outcomes before treating any indicator as evidence
   of actual fraud.

### Analytical Limitations

- Fraud flags are not confirmed fraud outcomes.
- Multiple claims from the same customer may affect
  the independence assumptions of statistical tests.
- The analyses identify associations, not causation.
- The prioritization results are retrospective and
  do not establish predictive performance.

### Conclusion

Customer risk scores and risk categories show the
clearest observed relationship with fraud flags in
this dataset.

A risk-based review strategy may help concentrate
investigation resources on claims with higher
observed flag rates while supporting consistent,
evidence-based claim handling.
